# Causal Backtest & Statistical Validation

Rebuilds the statistical arbitrage signal without look-ahead and tests the
result against a null, correcting the three defects documented in
`results/BACKTEST_FINDINGS.md`.

The original pipeline traded the cumulative sum of the Kalman filter's
one-step innovations. Its update step, `beta_t = beta_pred + K e_t`, leaves
a `-H_{t+1} K e_t` term in the next innovation, manufacturing negative
autocorrelation in proportion to the Kalman gain — which the strategy then
traded as if it were a market effect. It also calibrated the OU process and
selected the tradeable universe on the full sample.

Here, betas are fit on a trailing window ending strictly before the scored
day, residuals are cumulated within that window, the OU process is
calibrated on that window alone, and the traded return is formed
out-of-sample.

## 1: Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.backtest import PortfolioBacktester
from src.causal_signal import CausalSignalEngine
from src.data_loader import fetch_equity_returns
from src.rolling_engine import RollingPCAEngine
from src.validation import residual_autocorrelation_check, sign_shuffle_null

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)

## 2: Data & walk-forward factor extraction

In [ ]:
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "AMD", "INTC", "QCOM",
    "JPM", "BAC", "WFC", "C", "GS", "MS", "XOM", "CVX", "COP", "SLB",
]
returns = fetch_equity_returns(tickers, start_date="2019-01-01", end_date="2025-01-01")

rolling_pca = RollingPCAEngine(n_components=5, window=504, rebalance_freq=21)
factor_returns = rolling_pca.fit_transform(returns)

returns = returns.loc["2023-01-01":]
factor_returns = factor_returns.loc["2023-01-01":]
print(f"Backtest period: {returns.index[0].date()} to {returns.index[-1].date()} "
      f"({len(returns)} days x {returns.shape[1]} assets)")

## 3: Causal signal construction

`CausalSignalEngine` returns the s-scores and, separately, the idiosyncratic
returns the book actually earns — formed by applying betas fit strictly in
the past to the scored day's realized factor returns.

In [ ]:
engine = CausalSignalEngine(lookback=60)
s_scores, oos_residuals, sigma_eq = engine.compute(returns, factor_returns)

check = residual_autocorrelation_check(oos_residuals, returns)
print("Lag-1 autocorrelation")
print(f"  raw daily returns      : {check['raw_lag1_autocorr']:+.4f}")
print(f"  traded OOS residuals   : {check['traded_lag1_autocorr']:+.4f}")
print(f"  excess (induced)       : {check['excess']:+.4f}")
print(f"  within tolerance       : {check['within_tolerance']}")
print("\nFor reference, the original Kalman innovations showed -0.1132 —")
print("structure the estimator created rather than found.")

## 4: Backtest

In [ ]:
backtester = PortfolioBacktester(
    s_open=1.25, s_close=0.5, transaction_cost_bps=5.0, max_gross_leverage=1.0
)
median_sigma = sigma_eq.median().to_dict()

signals = backtester.generate_signals(s_scores.fillna(0.0))
weights = backtester.compute_portfolio_weights(signals, median_sigma)
results = backtester.run_backtest(weights, oos_residuals.fillna(0.0))

metrics = backtester.calculate_metrics(results["net_returns"])
print("=== Causal pipeline, net of 5 bps ===")
for k, v in metrics.items():
    print(f"  {k:24s} {v: .4f}")

## 5: Sign-shuffle null

Flipping the sign of each traded return preserves volatility and tail shape
while destroying any genuine link between signal and outcome. A strategy
with real edge scores far above this distribution.

In [ ]:
def sharpe_of(traded: pd.DataFrame) -> float:
    w = backtester.compute_portfolio_weights(
        backtester.generate_signals(s_scores.fillna(0.0)), median_sigma
    )
    r = backtester.run_backtest(w, traded.fillna(0.0))
    return backtester.calculate_metrics(r["net_returns"])["Sharpe Ratio"]

null = sign_shuffle_null(sharpe_of, oos_residuals, n_draws=200, seed=0)
print("=== Sign-shuffle null (200 draws) ===")
print(f"  observed Sharpe   : {null['observed_sharpe']:+.3f}")
print(f"  null mean (sd)    : {null['null_mean']:+.3f} ({null['null_sd']:.3f})")
print(f"  null 95th pct     : {null['null_p95']:+.3f}")
print(f"  z-score           : {null['z_score']:+.2f}")
print(f"  empirical p-value : {null['p_value']:.3f}")
print(f"  significant       : {null['significant']}")

## 6: Conclusion

Under a look-ahead-free construction the induced autocorrelation disappears
and the strategy's Sharpe falls from 8.59 to roughly 0.5, which the
sign-shuffle null cannot distinguish from luck (p ≈ 0.11).

**The correct reportable outcome for this universe and period is a null
result from a well-specified pipeline, not the Sharpe the original
construction produced.** Paths worth exploring before concluding no edge
exists: a larger universe, intraday or weekly horizons, a borrow/shorting
cost model, and explicit factor-hedge slippage.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True,
                               gridspec_kw={"height_ratios": [2.5, 1.5]})
ax1.plot(results.index, results["net_equity"], color="navy", lw=2,
         label=f"Causal pipeline (Sharpe {metrics['Sharpe Ratio']:.2f})")
ax1.axhline(1.0, color="black", lw=0.8, ls=":")
ax1.set_ylabel("Portfolio Value (Base 1.0)")
ax1.set_title("Look-Ahead-Free Statistical Arbitrage Backtest")
ax1.legend(loc="upper left")

dd = (results["net_equity"] - results["net_equity"].cummax()) / results["net_equity"].cummax()
ax2.fill_between(dd.index, dd, 0, color="crimson", alpha=0.3)
ax2.plot(dd.index, dd, color="crimson", lw=1)
ax2.set_ylabel("Drawdown"); ax2.set_xlabel("Date")
plt.tight_layout(); plt.show()